# Modeling
Let's begin by initializing and preparing our data as before. Note that this type of machine learning is known as *supervised machine learning* because it requires correctly labelled data to serve as our training dataset.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Reload and prepare data
df = pd.read_csv('../data/cumulative.csv')
df = df[df['koi_disposition'] != 'CANDIDATE']
df['label'] = (df['koi_disposition'] == 'CONFIRMED').astype(int)

features = ['koi_period', 'koi_depth', 'koi_duration', 'koi_prad',
            'koi_teq', 'koi_impact', 'koi_steff', 'koi_slogg', 'koi_srad']

log_features = ['koi_period', 'koi_depth', 'koi_duration', 'koi_prad', 'koi_srad']

X = df[features].copy()
for col in log_features:
    X[col] = np.log1p(X[col])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Logistic Regression Baseline
Let's consider a xy-plane in cartesian coordinates.

Our dataset contains only two values: confirmed exoplanet (1) and false positives (0). This will exist on our Y-axis and the X-axis is all of our inputs (i.e the data's features). How can we predict based on a given input that it is either a confirmed exoplanet, a 1, or a false positive, a 0? We can choose a method like *linear regression* (a.k.a a straight line of best fit).

A linear regression model will do its prediction as a linear function of our features plus a *bias*, the difference between the expected value of the estimator and the true value. Suppose we have a scattor plot showing a pass or fail on a test on the y-axis and the number of hours studied on the x-axis. The number of hours is a feature so the model could be:
 $$y = a_1*(number \, of \, hours) + bias$$

Say we also include the number of coffee cups drank, then the model could be:
$$y = a_1*(number \, of \, hours) + a_2*(number \, of \, coffee \, cups \, drank) + bias$$

Ultimately, by tweaking the constants and the bias, we can find the line of best fit. The domain and range of this model can be said as follows:
$$x\in[-\infty, \infty]$$
$$y\in[-\infty, \infty]$$

But what if the *y* value is limited as such?
$$y\in[0, 1]$$

This is where a logistic regression can come in. It squashes the regression, our prediction model, between the domain *[0,1]* by applying a sigmoid function instead. 

**Sigmoid Function:**
$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

It'll do the same as before like a linear regression model but it will be more accurate. scikit-learn already has a function that lets us do this so let's use that.

In [11]:
lr_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', random_state=42))
])

lr_pipeline.fit(X_train, y_train)
lr_predictions = lr_pipeline.predict(X_test)

print("Logistic Regression Results:")
print(f"Accuracy: {accuracy_score(y_test, lr_predictions):.4f}")
print(f"\n{classification_report(y_test, lr_predictions, target_names=['False Pos', 'Planet'])}")

Logistic Regression Results:
Accuracy: 0.8087

              precision    recall  f1-score   support

   False Pos       0.94      0.77      0.85      1005
      Planet       0.64      0.89      0.75       459

    accuracy                           0.81      1464
   macro avg       0.79      0.83      0.80      1464
weighted avg       0.85      0.81      0.82      1464



This is the simplest model we can use, and it is decently accurate with a percentage of 80.87%.

# Random Forest
There is a classification model known as the *random forest* model. This model is based on the concept of a *decision tree classification* so let's begin by understanding that.

A decision tree is a series of "yes" or "no" questions based on the features provided on the dataset. It's like a binary search tree data structure but a little different. Each node that has those "yes" or "no" questions are known as the *decision nodes* and nodes with no child nodes are called *leaf nodes*. 

A decision tree takes each individual feature and lines up all of the data accordingly, looks at its label, and finds the cut off to best seperate the data. It repeats this for every feature found that we want to use.

A random forest classifier has multiple decision trees utilizing a subset of the original training dataset. It will randomly choose two features to base off of. Now, when we have new data, it will run through all of these decision trees and which label appears the most often is the final result.

In [4]:
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=200, class_weight='balanced', random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_predictions = rf_pipeline.predict(X_test)

print("Random Forest Results:")
print(f"Accuracy: {accuracy_score(y_test, rf_predictions):.4f}")
print(f"\n{classification_report(y_test, rf_predictions, target_names=['False Pos', 'Planet'])}")

Random Forest Results:
Accuracy: 0.9003

              precision    recall  f1-score   support

   False Pos       0.92      0.94      0.93      1005
      Planet       0.86      0.81      0.84       459

    accuracy                           0.90      1464
   macro avg       0.89      0.88      0.88      1464
weighted avg       0.90      0.90      0.90      1464



# Gradient Boosting
Gradient boosting also relies on decision trees but it functions differently from the random forest method. This method works by starting off with one tree that tries to label all of the training dataset. For every label that was predicted wrong, we then create another training set based on tho incorrectly predicted data and tries to learn why the original tree made its mistake. When the second tree makes mistakes, we create a third tree and so on until we have an accurate model. This is known as *Gradient Boosting*

In [5]:
gb_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.1, max_depth=5, random_state=42
    ))
])

gb_pipeline.fit(X_train, y_train)
gb_predictions = gb_pipeline.predict(X_test)

print("Gradient Boosting Results:")
print(f"Accuracy: {accuracy_score(y_test, gb_predictions):.4f}")
print(f"\n{classification_report(y_test, gb_predictions, target_names=['False Pos', 'Planet'])}")

Gradient Boosting Results:
Accuracy: 0.9173

              precision    recall  f1-score   support

   False Pos       0.94      0.94      0.94      1005
      Planet       0.87      0.87      0.87       459

    accuracy                           0.92      1464
   macro avg       0.90      0.90      0.90      1464
weighted avg       0.92      0.92      0.92      1464



# Cross Validation

Let's try comparing each of the different models now!

In [7]:
models = {
    'Logistic Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(
            n_estimators=200, class_weight='balanced', random_state=42
        ))
    ]),
    'Gradient Boosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.1, max_depth=5, random_state=42
        ))
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1')
    print(f"{name}: F1 = {scores.mean():.4f} (+/- {scores.std():.4f})")

Logistic Regression: F1 = 0.7319 (+/- 0.0082)
Random Forest: F1 = 0.8398 (+/- 0.0154)
Gradient Boosting: F1 = 0.8668 (+/- 0.0053)


Looking at this output, we can rank each model based on how accurate it is. What we get is the following:
$$Gradient \; Boosting > Random \; Forest > Logistic \; Regression$$
This means for tuning and evaluating, we're going to use the Gradient Boosting model.